# Model 2: Isolation Forest for Anomaly Detection

This notebook implements an unsupervised Isolation Forest algorithm as a secondary security layer to detect novel or "zero-day" fraudulent financial transactions.

The model is trained using the processed ULB credit card fraud dataset and rigorously evaluated across multiple environments (ULB, IEEE-CIS, and PaySim datasets) using standard classification metrics such as:

- Precision
- Recall
- F1-score
- Confusion Matrix

In [16]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def load_data(file_name):
    
    paths_to_try = [
        f'../../data/processed/{file_name}',  
        f'data/processed/{file_name}',        
        f'../data/processed/{file_name}'      
    ]
    
    for p in paths_to_try:
        if os.path.exists(p):
            print(f"Found and successfully loaded: {p}")
            return pd.read_csv(p)
            
    
    print(f"Could not find {file_name}!")
    print(f"Your Jupyter environment is currently running from: {os.getcwd()}")
    return None


print("Searching for and loading datasets...")

df = load_data('processed_ulb_dataset.csv')
df_ieee = load_data('processed_ieee_dataset.csv')
df_paysim = load_data('processed_paysim_dataset.csv')

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


print("Training Isolation Forest model (Model 2)...")


iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)


iso_forest.fit(X_train)


print("Predicting results...")
y_pred = iso_forest.predict(X_test)
y_pred_binary = np.where(y_pred == -1, 1, 0)

print("\n=== EVALUATION RESULTS (MODEL 2: ISOLATION FOREST) ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))


cm = confusion_matrix(y_test, y_pred_binary)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')

plt.title('Isolation Forest - Confusion Matrix')
plt.ylabel('Actual (0: Normal, 1: Fraud)')
plt.xlabel('Predicted (0: Normal, 1: Fraud)')

plt.show()

In [ ]:
X_ieee = df_ieee.drop('isFraud', axis=1)
y_ieee = df_ieee['isFraud']


X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_ieee, y_ieee, test_size=0.2, random_state=42
)

print("Training Isolation Forest model (IEEE dataset)...")

iso_forest_ieee = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)

iso_forest_ieee.fit(X_train_i)

print("Predicting results...")

y_pred_i = iso_forest_ieee.predict(X_test_i)
y_pred_binary_i = np.where(y_pred_i == -1, 1, 0)

print("\n=== EVALUATION RESULTS (IEEE DATASET) ===")

print("Confusion Matrix:\n", confusion_matrix(y_test_i, y_pred_binary_i))
print("\nClassification Report:\n", classification_report(y_test_i, y_pred_binary_i))


cm_ieee = confusion_matrix(y_test_i, y_pred_binary_i)

plt.figure(figsize=(6,4))
sns.heatmap(cm_ieee, annot=True, fmt='d', cmap='Greens')

plt.title('Isolation Forest - IEEE Dataset')
plt.ylabel('Actual (0: Normal, 1: Fraud)')
plt.xlabel('Predicted (0: Normal, 1: Fraud)')

plt.show()

In [ ]:
X_pay = df_paysim.drop('isFraud', axis=1)
y_pay = df_paysim['isFraud']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pay, y_pay, test_size=0.2, random_state=42
)

print("Training Isolation Forest model (PaySim dataset)...")

iso_forest_pay = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42
)

iso_forest_pay.fit(X_train_p)

print("Predicting results...")

y_pred_p = iso_forest_pay.predict(X_test_p)
y_pred_binary_p = np.where(y_pred_p == -1, 1, 0)

print("\n=== EVALUATION RESULTS (PAYSIM DATASET) ===")

print("Confusion Matrix:\n", confusion_matrix(y_test_p, y_pred_binary_p))
print("\nClassification Report:\n", classification_report(y_test_p, y_pred_binary_p))


cm_pay = confusion_matrix(y_test_p, y_pred_binary_p)

plt.figure(figsize=(6,4))
sns.heatmap(cm_pay, annot=True, fmt='d', cmap='Purples')

plt.title('Isolation Forest - PaySim Dataset')
plt.ylabel('Actual (0: Normal, 1: Fraud)')
plt.xlabel('Predicted (0: Normal, 1: Fraud)')

plt.show()

In [ ]:
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


cm = confusion_matrix(y_test, y_pred_binary)

metrics = {
    "accuracy": round(float(accuracy_score(y_test, y_pred_binary)), 4),
    "precision": round(float(precision_score(y_test, y_pred_binary, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test, y_pred_binary, zero_division=0)), 4),
    "f1_score": round(float(f1_score(y_test, y_pred_binary, zero_division=0)), 4),
    "confusion_matrix": {
        "true_negatives": int(cm[0,0]),
        "false_positives": int(cm[0,1]),
        "false_negatives": int(cm[1,0]),
        "true_positives": int(cm[1,1])
    }
}

model2_json = {
  "model_info": {
    "model_name": "Isolation Forest",
    "model_type": "Unsupervised Learning",
    "algorithm": "Isolation Forest",
    "developer": "Ho Cao Cuong - 105687343"
  },
  "hyperparameters": {
    "n_estimators": 100,
    "contamination": 0.01,
    "random_state": 42
  },
  "datasets_used": [
    "ULB Credit Card Fraud",
    "IEEE-CIS Fraud Detection",
    "PaySim Mobile Money"
  ],
  "performance_metrics": {
    "ULB_Dataset": metrics
  },
  "saved_artifacts": {
    "model_file": "None (Unsupervised model dynamically fitted)",
    "notebook_file": "Model2_IsolationForest.ipynb"
  }
}


with open("model2_info.json", "w", encoding="utf-8") as f:
    json.dump(model2_json, f, indent=4)

print("✅ Successfully created the file 'model2_info.json'!")